# 🌊 SeaSentinel: High-Accuracy Sonar Marine Debris Model Training
### End-to-End Deep Learning Pipeline with GPU Acceleration, Advanced Augmentation, and Multi-Metric Accuracy Evaluation

This notebook trains and evaluates state-of-the-art neural networks for detecting and classifying marine debris, ghost nets, shipwrecks, and subsea hazards in acoustic sonar imagery.

---

## ⚙️ Step 1: Install Dependencies & Verify GPU Accelerator
In Google Colab, select: **Runtime $\to$ Change runtime type $\to$ T4 GPU**.

In [ ]:
!pip install -q kagglehub timm albumentations scikit-learn seaborn matplotlib

import os
import sys
import json
import time
import random
import shutil
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set reproducible seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Compute Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"✓ GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 📥 Step 2: Automatic Dataset Ingestion (Kaggle FLS Marine Debris)

In [ ]:
def download_and_setup_dataset() -> Path:
    import kagglehub
    print("Downloading Forward-Looking Sonar Marine Debris Dataset from Kaggle...")
    raw_path = kagglehub.dataset_download("era2730/forward-looking-sonar-marine-debris-dataset")
    root = Path(raw_path)
    print(f"Dataset downloaded to: {root}")
    
    # Auto-detect folder root containing classes
    IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        dirpath = Path(dirpath)
        if not dirnames:
            continue
        class_count = 0
        for d in dirnames:
            sub = dirpath / d
            if any(p.suffix.lower() in IMG_EXTS for p in sub.iterdir() if p.is_file()):
                class_count += 1
        if class_count >= 2:
            candidates.append((dirpath, class_count))
            
    if candidates:
        candidates.sort(key=lambda x: (len(x[0].parts), -x[1]))
        return candidates[0][0]
    return root

DATA_DIR = download_and_setup_dataset()
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"\n✓ Detected {len(classes)} Marine Debris Classes:")
total_images = 0
for c in classes:
    count = len(list((DATA_DIR / c).glob("*")))
    total_images += count
    print(f"  • {c:<22}: {count} images")
print(f"\nTotal dataset images: {total_images}")

## 🔬 Step 3: Advanced Sonar Acoustic Data Augmentation
To maximize accuracy and prevent overfitting on small datasets, we apply specialized acoustic transformations:
- **Speckle Noise Injection**: Simulates multi-path acoustic scattering
- **Contrast / Brightness Jitter**: Simulates variable water column turbidity
- **Random Affine & Horizontal Flips**: Rotation invariance for drifting debris

In [ ]:
IMG_SIZE = 224

class AddAcousticSpeckleNoise(object):
    """Simulates multiplicative acoustic speckle noise."""
    def __init__(self, variance=0.04):
        self.variance = variance
        
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * np.sqrt(self.variance)
        noisy_tensor = tensor + tensor * noise
        return torch.clamp(noisy_tensor, 0.0, 1.0)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    AddAcousticSpeckleNoise(variance=0.03),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SonarDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int]], class_to_idx: Dict[str, int], transform=None):
        self.samples = samples
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# Load full sample list
class_to_idx = {c: i for i, c in enumerate(classes)}
all_samples = []
for c in classes:
    for p in (DATA_DIR / c).glob("*"):
        if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp", ".tif"}:
            all_samples.append((str(p), class_to_idx[c]))

# Split: 70% Train, 15% Validation, 15% Test
n_total = len(all_samples)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_sub, val_sub, test_sub = random_split(
    all_samples, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_dataset = SonarDataset([all_samples[i] for i in train_sub.indices], class_to_idx, transform=train_transform)
val_dataset = SonarDataset([all_samples[i] for i in val_sub.indices], class_to_idx, transform=eval_transform)
test_dataset = SonarDataset([all_samples[i] for i in test_sub.indices], class_to_idx, transform=eval_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✓ Training Samples:   {len(train_dataset)}")
print(f"✓ Validation Samples: {len(val_dataset)}")
print(f"✓ Testing Samples:    {len(test_dataset)}")

## 🧠 Step 4: High-Accuracy Neural Network Architecture with Attention
We use a pre-trained **ResNet-34 / ResNet-18** backbone with unfreezed deep layers, **Squeeze-and-Excitation Channel Attention**, and a multi-layer classifier head with **BatchNorm** and **Dropout**.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation Attention Channel Recalibration."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(x).view(b, c, 1, 1)
        return x * w

class HighAccuracySonarNet(nn.Module):
    def __init__(self, num_classes=len(classes), freeze_early_layers=True):
        super().__init__()
        base = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        
        if freeze_early_layers:
            for param in base.parameters():
                param.requires_grad = False
            # Unfreeze layers 3 and 4 for domain adaptation
            for param in base.layer3.parameters():
                param.requires_grad = True
            for param in base.layer4.parameters():
                param.requires_grad = True
                
        self.features = nn.Sequential(
            base.conv1,
            base.bn1,
            base.relu,
            base.maxpool,
            base.layer1,
            base.layer2,
            base.layer3,
            base.layer4,
            SEBlock(512)
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        logits = self.classifier(x)
        return logits

model = HighAccuracySonarNet(num_classes=len(classes)).to(DEVICE)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model Initialized: {trainable_params:,} Trainable Parameters")

## 🎯 Step 5: Class-Weighted Loss Function & Cosine Annealing Optimizer
We compute inverse class frequencies to heavily penalize errors on rare debris categories, and use **Label Smoothing (0.1)** to prevent overconfident misclassifications.

In [ ]:
# Compute class weights to handle severe class imbalance
train_labels = [label for _, label in train_dataset.samples]
counts = Counter(train_labels)
class_weights = torch.tensor(
    [1.0 / max(1, counts[i]) for i in range(len(classes))],
    dtype=torch.float32
).to(DEVICE)
class_weights = class_weights / class_weights.sum() * len(classes)

# Criterion: Label-Smoothed Weighted Cross-Entropy
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Optimizer: AdamW with weight decay
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

# Scheduler: Cosine Annealing with Warm Restarts
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print("✓ Loss Function & Cosine Annealing Scheduler Ready")

## 🚀 Step 6: Model Training & Validation Loop
Executes full training with early stopping, saving the best checkpoint to `best_model.pt`.

In [ ]:
EPOCHS = 25
PATIENCE = 7
OUTPUT_DIR = "training_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_val_loss = float("inf")
best_val_acc = 0.0
epochs_no_improve = 0

history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": []
}

print("=" * 75)
print(f"  Starting Training for {EPOCHS} Epochs on {DEVICE}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        
        train_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    scheduler.step()
    
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)
    
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {epoch_train_loss:.4f}  Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f}  Acc: {epoch_val_acc*100:.2f}%")
          
    # Checkpoint on best validation accuracy / loss
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": classes,
            "best_val_acc": best_val_acc,
            "img_size": IMG_SIZE
        }, os.path.join(OUTPUT_DIR, "best_model.pt"))
        print(f"  ⭐ New Best Model Saved! (Val Accuracy: {best_val_acc*100:.2f}%)")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered after {PATIENCE} epochs with no improvement.")
            break

print("\n✓ Training Complete!")

## 📊 Step 7: Comprehensive Accuracy & Metrics Evaluation
We evaluate the model on the held-out **Test Set** (never seen during training):
- Overall Accuracy & Top-3 Accuracy
- Per-Class Precision, Recall, and F1-Score
- High-Resolution Confusion Matrix Heatmap
- Loss and Accuracy Convergence Plots

In [ ]:
# Load best weights
ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
top3_correct = 0
total_test = 0

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        
        # Top 1
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Top 3
        top3_preds = outputs.topk(3, dim=1).indices
        for i in range(labels.size(0)):
            if labels[i] in top3_preds[i]:
                top3_correct += 1
        total_test += labels.size(0)

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
top3_acc = top3_correct / total_test

print("=" * 75)
print(f"  FINAL TEST ACCURACY (Top-1): {test_acc*100:.2f}%")
print(f"  FINAL TOP-3 ACCURACY:        {top3_acc*100:.2f}%")
print("=" * 75)

# Classification Report
print("\n--- Detailed Classification Metrics ---")
report = classification_report(all_labels, all_preds, target_names=classes, digits=4)
print(report)

with open(os.path.join(OUTPUT_DIR, "test_classification_report.txt"), "w") as f:
    f.write(report)

## 📈 Step 8: Visual Accuracy Plots & Confusion Matrix

In [ ]:
# 1. Plot Training & Validation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", color="#EF4444", linewidth=2)
axes[0].plot(history["val_loss"], label="Validation Loss", color="#38BDF8", linewidth=2)
axes[0].set_title("Convergence Loss Curves", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Cross-Entropy Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot([a * 100 for a in history["train_acc"]], label="Train Accuracy", color="#10B981", linewidth=2)
axes[1].plot([a * 100 for a in history["val_acc"]], label="Val Accuracy", color="#00E599", linewidth=2)
axes[1].set_title("Model Accuracy (%) Progression", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy_curves.png"), dpi=200)
plt.show()

# 2. Plot High-Resolution Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, cbar=True)
plt.title(f"Acoustic Debris Confusion Matrix (Test Accuracy: {test_acc*100:.2f}%)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Label", fontsize=11, labelpad=10)
plt.ylabel("True Ground Truth Label", fontsize=11, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_heatmap.png"), dpi=200)
plt.show()

## 📦 Step 9: Export to ONNX INT8 for Edge Marine Drones & Download Checkpoint

In [ ]:
# Export to ONNX
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
onnx_path = os.path.join(OUTPUT_DIR, "best_model.onnx")

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}}
)
print(f"✓ ONNX Model Exported to: {onnx_path}")

# Create ZIP bundle of model & reports for easy download
bundle_zip = "trained_model_and_reports.zip"
shutil.make_archive("trained_model_and_reports", "zip", OUTPUT_DIR)
print(f"✓ Downloadable Bundle Created: {bundle_zip}")

# In Google Colab, trigger automatic download
try:
    from google.colab import files
    files.download(bundle_zip)
    print("Triggered automatic browser download in Colab!")
except ImportError:
    print(f"File ready at: {os.path.abspath(bundle_zip)}")